# /classify — Endpoint Evaluation

Checks topic assignment, out-of-scope detection, and geo-scope classification.

The fixture `text` field is passed directly as `summary` to `/classify` (sufficient for
classification quality testing without requiring the full summarizer pipeline).

**Prerequisite:** NLP service running. `/readyz` → 200.

In [ ]:
import sys, time, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, keyword_hit_rate, NLP_BASE_URL, HEADERS

cases = load_fixture('classify_cases.json')
print(f'Loaded {len(cases)} test cases')

In [ ]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
assert r.status_code == 200, f'Service not ready: {r.status_code} {r.text}'
print('Service ready')

In [ ]:
results = []

for case in cases:
    t0 = time.monotonic()
    resp = requests.post(
        f'{NLP_BASE_URL}/classify',
        json={
            'article_id':   case['article_id'],
            'summary':      case['text'],          # article text used as proxy summary
            'geo_cities':   [],
            'search_tags':  [],
            'source_profile': None,
        },
        headers=HEADERS,
    )
    latency = time.monotonic() - t0
    assert resp.status_code == 200, \
        f"{case['article_id']}: HTTP {resp.status_code} — {resp.text}"
    data = resp.json()

    scope_ok      = (data['geo_scope'] == case['expected_scope_signal']) \
                    if case.get('expected_scope_signal') else True
    oos_ok        = data['out_of_scope'] == case['expected_out_of_scope']
    expected_kws  = case.get('expected_topics_include', [])
    topics_text   = ' '.join(data['topics'])
    topic_hit     = keyword_hit_rate(topics_text, expected_kws) if expected_kws else 1.0
    passed        = oos_ok and scope_ok and (topic_hit >= 0.5 or not expected_kws)
    icon          = '✅' if passed else '❌'

    results.append({
        'id':         case['article_id'],
        'oos_ok':     oos_ok,
        'scope_ok':   scope_ok,
        'topic_hit':  topic_hit,
        'latency_s':  latency,
        'pass':       passed,
    })

    print(f"{icon} [{case['article_id']}]  "
          f"out_of_scope={data['out_of_scope']} (expected {case['expected_out_of_scope']})  "
          f"scope={data['geo_scope']!r}  "
          f"topic_hit={topic_hit:.2f}  {latency:.2f}s")
    print(f"   topics: {data['topics']}")
    if not passed:
        print(f"   FAIL: oos_ok={oos_ok}  scope_ok={scope_ok}")
    print()

In [ ]:
passing    = [r for r in results if r['pass']]
oos_pass   = sum(1 for r in results if r['oos_ok'])
scope_pass = sum(1 for r in results if r['scope_ok'])
avg_hit    = sum(r['topic_hit'] for r in results) / len(results)
avg_lat    = sum(r['latency_s'] for r in results) / len(results)

print_scorecard('/classify', {
    'Cases':                           len(results),
    'Passing (all checks)':            f'{len(passing)}/{len(results)}',
    'Out-of-scope accuracy':           f'{oos_pass}/{len(results)}',
    'Scope signal accuracy':           f'{scope_pass}/{len(results)}',
    'Avg topic keyword hit rate':      avg_hit,
    'Avg latency (s)':                 avg_lat,
})